# Pipeline Node Agents Experiments

This notebook demonstrates how to run the Pipeline Node Agents framework with locally hosted LLMs (Qwen3:8b and Llama3.2) using Ollama in Google Colab.

**Contents:**
1. GPU and CUDA verification
2. Ollama installation and server setup
3. Model downloads (Qwen3:8b, Llama3.2)
4. Pipeline Node Agents setup
5. Model testing and pipeline execution

In [1]:
# Check GPU availability
!nvidia-smi

Wed Feb  4 12:16:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Check CUDA version
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0


## 1. Install Ollama

Install dependencies and the Ollama server.

In [3]:
# Install zstd (required dependency) and Ollama
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh -o install.sh
!chmod +x install.sh && ./install.sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 41 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 4s (159 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 121689 files and directories currently i

In [4]:
# Start Ollama server in background
import subprocess
import time

process = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)

if process.poll() is None:
    print("Ollama server started successfully.")
else:
    print("Ollama server failed to start.")
    stdout, stderr = process.communicate()
    print(f"Stdout: {stdout.decode()}")
    print(f"Stderr: {stderr.decode()}")

Ollama server started successfully.


## 2. Download Models

Pull Qwen3:8b and Llama3.2 models using Ollama.

In [5]:
# Download models
!ollama pull qwen3:8b
!ollama pull llama3.2:latest

In [6]:
# Verify models were downloaded
!ollama list

NAME               ID              SIZE      MODIFIED               
llama3.2:latest    a80c4f17acd5    2.0 GB    Less than a second ago    
qwen3:8b           500a1f067a9f    5.2 GB    25 seconds ago            


## 3. Set Up Pipeline Node Agents

Clone the repository and install dependencies.

In [7]:
# Clone repository and checkout master branch
!git clone https://github.com/ser0vs/pipeline_node_agents.git
!cd /content/pipeline_node_agents && git checkout master

Cloning into 'pipeline_node_agents'...
remote: Enumerating objects: 568, done.
remote: Counting objects: 100% (121/121), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 568 (delta 65), reused 75 (delta 33), pack-reused 447 (from 1)
Receiving objects: 100% (568/568), 422.63 KiB | 2.26 MiB/s, done.
Resolving deltas: 100% (330/330), done.
Already on 'master'
Your branch is up to date with 'origin/master'.


In [8]:
# Install Poetry and project dependencies
!pip install poetry
!cd /content/pipeline_node_agents && poetry install

Die letzten 5000 Zeilen der Streamingausgabe wurden abgeschnitten.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.8/287.8 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 340.9/340.9 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.7/78.7 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.8/453.8 kB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 144.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.0/469.0 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 129.7 MB/s eta 0:00:00
Creating virtualenv pipeline-node-agents-AK2flgi8-py3.12 in /root/.cache/pypoetry/virtualenvs
Installing dependencies from lock file

Package operations: 149 installs, 0 updates, 0 removals

  - Installing aiohappyeyebal

In [9]:
# Make scripts executable
!cd /content/pipeline_node_agents && chmod +x scripts/*

## 4. Configure API Key (Optional)

For web search functionality, set the SERPER_API_KEY using Colab's secret manager (🔑 icon in the left panel).

## 5. Test Model

Quick test to verify Qwen3 is working.

In [10]:
# Test model with a simple query
!ollama run qwen3:8b "Hello world!"

Thinking...
Okay, the user said "Hello world!" and I need to respond. Let me think about how to approach this.

First, "Hello world!" is a common greeting, so I should acknowledge it politely. Maybe start with a friendly greeting back. Then, since the user might be testing the system or just starting a conversation, I should offer assistance. I should keep the tone positive and open-ended. Let me make sure the response is welcoming and invites them to ask questions or share more. Also, check for any possible typos or errors in the response. Alright, that should cover it.
...done thinking.

Hello! 😊 How can I assist you today? Whether you have questions, need help with something, or just want to chat, I'm here for you! What's on your mind?



## 6. Run Pipeline (manunally in terminal)

Please open the **terminal** and copy-paste the code line by line.

First off, go to project directory in src:
```bash
cd /content/pipeline_node_agents/src/pipeline_node_agents
```

After it, there are several options to execute the pipeline.

**Option A:** Run directly with Poetry:
```bash
poetry run python3 examples/trip_planner/pipeline.py
```

**Option B:** Run single pipeline with custom input:
```bash
./scripts/run_single_pipeline.sh examples/trip_planner/pipeline.py 1 $'Madrid, Dubai\n17 January 2025\n25 January 2025'
```

**Option C:** Run smoke test for all pipelines:
```bash
./scripts/run_smoke_pipelines.sh 1
```

## 7. After execution (optional)

In [12]:
# Check logs output
!ls -lt /content/pipeline_node_agents/logs

total 8
drwxr-xr-x 2 root root 4096 Feb  4 12:24 trip_planner_pipeline
drwxr-xr-x 4 root root 4096 Feb  4 12:18 supplemental_materials
